# GridLock Hackathon 2.0 — Traffic Demand Prediction
## v6 — Clean Final Submission (LightGBM with fixed hyperparams)

- Train on FULL train (77,299 rows, day 48 + day 49)  
- Predict on test (41,778 rows)  
- Fixed hyperparams (no tuning)  
- Day 49 holdout for val reporting only

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
from lightgbm import LGBMRegressor, early_stopping as lgb_early_stop, log_evaluation as lgb_log
import pygeohash as pgh

pd.set_option('display.max_columns', 80)
np.random.seed(42)
print('Imports OK')

## Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Train days: {train["day"].value_counts().sort_index().to_dict()}')
print(f'Test  days: {test["day"].value_counts().sort_index().to_dict()}')

---
## Step 1: Feature Engineering

All stats fit on FULL train (both days). Applied uniformly to train and test.

In [ ]:
class FeatureEngineer:
    'v6 — simple, fits on full train.'

    @staticmethod
    def _decode_gh(gh):
        try:
            lat, lng, _, _ = pgh.decode_exactly(gh)
            return float(lat), float(lng)
        except Exception:
            return 0.0, 0.0

    @staticmethod
    def _parse_ts(ts):
        h, m = ts.split(':')
        return int(h), int(m)

    @staticmethod
    def _time_of_day(h):
        if   h <= 5:  return 0
        elif h <= 11: return 1
        elif h <= 17: return 2
        else:         return 3

    def fit(self, df):
        df = df.copy()
        # Pre-compute hour for group stats
        ts = df['timestamp'].apply(self._parse_ts)
        df['_hour'] = ts.apply(lambda x: x[0])

        # Temperature stats
        self._temp_med_global = float(df['Temperature'].median())
        self._temp_med_gh     = df.groupby('geohash')['Temperature'].median()
        self._temp_mean_gh    = df.groupby('geohash')['Temperature'].mean()
        self._global_temp_mean = float(df['Temperature'].mean())

        # Geohash frequency (count per geohash)
        self._gh_freq = df['geohash'].value_counts().astype(float)

        # Target encodings (from FULL train)
        self._global_demand = float(df['demand'].mean())
        alpha = 10

        # geohash_mean_demand (α=10 smoothed)
        gh_stats = df.groupby('geohash')['demand'].agg(['mean','count'])
        gh_stats['smooth'] = (gh_stats['count']*gh_stats['mean']
                              + alpha*self._global_demand) / (gh_stats['count'] + alpha)
        self._gh_mean_d = gh_stats['smooth'].to_dict()

        # geohash_hour_mean_demand (mean per (geohash, hour))
        self._gh_hour_mean_d = df.groupby(['geohash','_hour'])['demand'].mean().to_dict()

        # geohash_4_mean_demand (mean per geohash[:4])
        df['_gh4'] = df['geohash'].str[:4]
        self._gh4_mean_d = df.groupby('_gh4')['demand'].mean().to_dict()

        # Day 49 trajectory features
        d49 = df[df['day'] == 49]
        self._d49_h0 = d49[d49['_hour'] == 0].groupby('geohash')['demand'].mean().to_dict()
        self._d49_h1 = d49[d49['_hour'] == 1].groupby('geohash')['demand'].mean().to_dict()
        self._d49_h2 = d49[d49['_hour'] == 2].groupby('geohash')['demand'].mean().to_dict()

        self.fitted = True
        return self

    def transform(self, df):
        assert self.fitted, 'Call fit first'
        df = df.copy()

        # Timestamp features
        ts = df['timestamp'].apply(self._parse_ts)
        df['hour']         = ts.apply(lambda x: x[0])
        df['minute']       = ts.apply(lambda x: x[1])
        df['hour_sin']     = np.sin(2*np.pi*df['hour']/24)
        df['hour_cos']     = np.cos(2*np.pi*df['hour']/24)
        df['is_rush_hour'] = df['hour'].isin([7,8,9,17,18,19]).astype(int)
        df['time_of_day']  = df['hour'].apply(self._time_of_day)

        # Geohash decode
        geo = df['geohash'].apply(self._decode_gh)
        df['geohash_lat'] = geo.apply(lambda x: x[0])
        df['geohash_lng'] = geo.apply(lambda x: x[1])

        # Temperature: impute then deviation
        df['Temperature'] = df['Temperature'].fillna(
            df['geohash'].map(self._temp_med_gh).fillna(self._temp_med_global))
        df['temp_deviation'] = df['Temperature'] - df['geohash'].map(
            self._temp_mean_gh).fillna(self._global_temp_mean)

        # Binary categoricals
        df['LargeVehicles'] = (df['LargeVehicles'] == 'Allowed').astype(int)
        df['Landmarks']     = (df['Landmarks']     == 'Yes').astype(int)

        # One-hot: RoadType (Residential, Street, Highway, Unknown=NaN)
        for rt in ['Residential', 'Street', 'Highway']:
            df[f'RoadType_{rt}'] = (df['RoadType'] == rt).astype(int)
        df['RoadType_Unknown'] = df['RoadType'].isna().astype(int)

        # One-hot: Weather (Sunny, Rainy, Foggy, Snowy, Unknown=NaN)
        for wx in ['Sunny', 'Rainy', 'Foggy', 'Snowy']:
            df[f'Weather_{wx}'] = (df['Weather'] == wx).astype(int)
        df['Weather_Unknown'] = df['Weather'].isna().astype(int)

        # Target encodings
        df['geohash_mean_demand'] = df['geohash'].map(
            self._gh_mean_d).fillna(self._global_demand)

        gh_hr_keys = list(zip(df['geohash'], df['hour']))
        df['geohash_hour_mean_demand'] = [
            self._gh_hour_mean_d.get(k, np.nan) for k in gh_hr_keys]
        df['geohash_hour_mean_demand'] = df['geohash_hour_mean_demand'].fillna(
            df['geohash_mean_demand'])

        gh4 = df['geohash'].str[:4]
        df['geohash_4_mean_demand'] = gh4.map(
            self._gh4_mean_d).fillna(self._global_demand)

        df['geohash_frequency'] = df['geohash'].map(self._gh_freq).fillna(0.0)

        # Day 49 trajectory features (fill missing with geohash_mean_demand)
        fb = df['geohash_mean_demand']
        df['demand_hour0'] = df['geohash'].map(self._d49_h0).fillna(fb)
        df['demand_hour1'] = df['geohash'].map(self._d49_h1).fillna(fb)
        df['demand_hour2'] = df['geohash'].map(self._d49_h2).fillna(fb)
        df['demand_day49_trend'] = df['demand_hour1'] - df['demand_hour0']

        return df

print('FeatureEngineer v6 defined.')

In [ ]:
fe = FeatureEngineer().fit(train)
train_fe = fe.transform(train)
test_fe  = fe.transform(test)
print(f'Train FE: {train_fe.shape}  |  Test FE: {test_fe.shape}')
print(f'Day 49 trajectory coverage: '
      f'hour0={len(fe._d49_h0)}, hour1={len(fe._d49_h1)}, hour2={len(fe._d49_h2)} geohashes')

---
## Step 2: Feature Columns

In [ ]:
EXCLUDE = {'Index', 'geohash', 'timestamp', 'demand', 'RoadType', 'Weather'}
feature_cols = [c for c in train_fe.columns if c not in EXCLUDE]
print(f'Total features: {len(feature_cols)}')
for i, c in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {c}')

In [ ]:
X_all = train_fe[feature_cols].values.astype(np.float32)
y_all = train_fe['demand'].values.astype(np.float32)

nan_cnt = int(np.isnan(X_all).sum())
if nan_cnt > 0:
    X_all = np.nan_to_num(X_all, nan=0.0)
    print(f'Replaced {nan_cnt} NaN in X_all with 0')
print(f'X_all: {X_all.shape}  |  y_all: {y_all.shape}')
print(f'y range: [{y_all.min():.4f}, {y_all.max():.4f}]  mean: {y_all.mean():.4f}')

---
## Step 3: Validation Split (Reporting Only)

Train on day 48, validate on day 49 — purely to report val R² and train-val gap.  
Final submission model is retrained on FULL train.

In [ ]:
is_d48 = (train_fe['day'] == 48).values
tr_idx = np.where( is_d48)[0]
va_idx = np.where(~is_d48)[0]

X_train, y_train = X_all[tr_idx], y_all[tr_idx]
X_val,   y_val   = X_all[va_idx], y_all[va_idx]

print(f'Train (day 48): {X_train.shape}')
print(f'Val   (day 49): {X_val.shape}')

---
## Step 4: Train Val Model — Fixed Hyperparams

In [ ]:
PARAMS = dict(
    learning_rate     = 0.06654953824995291,
    num_leaves        = 496,
    min_child_samples = 100,
    subsample         = 0.9741638220848565,
    subsample_freq    = 1,
    colsample_bytree  = 0.8636008886793652,
    reg_alpha         = 0.3067673292781943,
    reg_lambda        = 1.7513019271871575,
    n_estimators      = 2000,
    random_state      = 42,
    n_jobs            = 4,
    verbose           = -1,
)

val_model = LGBMRegressor(**PARAMS)
val_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb_early_stop(50, verbose=False), lgb_log(200)]
)

val_train_r2 = r2_score(y_train, val_model.predict(X_train))
val_val_r2   = r2_score(y_val,   val_model.predict(X_val))
best_iter    = val_model.best_iteration_

print(f'Train R² = {val_train_r2:.4f}')
print(f'Val   R² = {val_val_r2:.4f}')
print(f'Gap      = {val_train_r2 - val_val_r2:.4f}')
print(f'Score    = {max(0, val_val_r2*100):.2f}')
print(f'Best iter: {best_iter}')

---
## Step 5: Retrain on FULL Train

n_estimators = best_iter (from val early-stopping) + 20% buffer.

In [ ]:
final_iters = int(best_iter * 1.20) + 1
print(f'Retraining on {X_all.shape[0]} rows with {final_iters} trees...')

final_params = {k: v for k, v in PARAMS.items() if k != 'n_estimators'}
final_params['n_estimators'] = final_iters

final_model = LGBMRegressor(**final_params)
final_model.fit(X_all, y_all)
print('Final model trained on full data.')

---
## Step 6: Predictions + Submission

In [ ]:
X_test = test_fe[feature_cols].values.astype(np.float32)
nan_te = int(np.isnan(X_test).sum())
if nan_te > 0:
    X_test = np.nan_to_num(X_test, nan=0.0)
    print(f'Replaced {nan_te} NaN in X_test with 0')
print(f'X_test: {X_test.shape}')

final_preds = final_model.predict(X_test)
final_preds = np.clip(final_preds, 0.0, 1.0)

print(f'\nPrediction stats:')
print(f'  min  : {final_preds.min():.6f}')
print(f'  max  : {final_preds.max():.6f}')
print(f'  mean : {final_preds.mean():.6f}')
print(f'  std  : {final_preds.std():.6f}')
print(f'  pct zeros: {100*(final_preds == 0).mean():.1f}%')

In [ ]:
submission = pd.DataFrame({'Index': test['Index'], 'demand': final_preds})
assert submission.shape == (41778, 2), f'Shape mismatch: {submission.shape}'
submission.to_csv('submission.csv', index=False)

print(f'submission.csv saved  |  shape: {submission.shape}')
print('\nFirst 5 rows:')
print(submission.head())
print('\nDescribe:')
print(submission['demand'].describe())

---
## Step 7: Final Report

In [ ]:
print('=' * 60)
print('  GRIDLOCK HACKATHON 2.0 — v6 FINAL SUBMISSION')
print('=' * 60)
print(f'Validation (day 48 → day 49):')
print(f'  Train R² = {val_train_r2:.4f}')
print(f'  Val   R² = {val_val_r2:.4f}')
print(f'  Gap      = {val_train_r2 - val_val_r2:.4f}')
print(f'  Score    = {max(0, val_val_r2*100):.2f}')
print(f'  Best iter (val): {best_iter}')

print(f'\nFinal submission model:')
print(f'  Trained on: {X_all.shape[0]} rows (full train)')
print(f'  n_estimators: {final_iters} (best_iter × 1.20)')
print(f'  Features: {len(feature_cols)}')

print(f'\nPrediction stats (clipped to [0, 1]):')
print(f'  min = {final_preds.min():.4f}  max = {final_preds.max():.4f}')
print(f'  mean = {final_preds.mean():.4f}  std = {final_preds.std():.4f}')

print(f'\nSubmission: submission.csv  shape={submission.shape}')
print('=' * 60)